# Laboratorio No 1

## Grupo Azul

---

### Daniel Felipe Sua Siempira
### Juan David Moreno D'Aleman

---

## 2. DESARROLLANDO: BÚSQUEDA SIN ADVERSARIO

### 2.1. A. MODELAR PROBLEMAS DE BÚSQUEDA

#### 2.1.1. Modelen las estructuras de datos auxiliares para implementar búsquedas en espacio de soluciones.

En base a la estructura dada en el enunciado, tenemos que finalizar la clase `Frontier`.

In [31]:
import heapq

class Node:
    def __init__(self, state, parent=None, action=None, cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost

    def __str__(self):
        return f"Node(state={self.state}, action={self.action}, cost={self.cost})"

class Frontier:
    def __init__(self, evaluation_function):
        self.evaluation_function = evaluation_function
        self.frontierNode = []
        self.cont = 0

    def is_empty(self):
        return len(self.frontierNode) == 0

    def pop(self):
        if self.is_empty():
            raise Exception("empty frontier")
        delete = heapq.heappop(self.frontierNode)
        return delete[2]

    def top(self):
        if self.is_empty():
            raise Exception("empty frontier")
        top = self.frontierNode[0][2]
        return top

    def add(self, node):
        valor = self.evaluation_function(node)
        heapq.heappush(self.frontierNode, (valor, self.cont, node))
        self.cont += 1

Para la clase `Frontier` le creamos una lista para guardar los nodos, sin embargo estos los guardamos en forma de tuplas, con 3 elementos, asi como un contador, este contador tiene la función de hacer de "indice" para la lista, ya que al ser una cola de prioridad basandose en el costo (por el momento no tomamos en cuenta la heuristica), en caso de que 2 nodos tengan el mismo coste, se elije por quien llego primero, para esto usamos el contador.

Por lo que las tuplas se crean en el orden `costo, contador, nodo`, para que primero compare el costo de ir a x nodo, en caso de ser igual se elegira por el quien llego o se leyo primero, o sea el contador, el cual al ser diferente para todos los elementos, siempre sera el elemento de desempate. Y finalmente nodo.

Para cada función en orden:

* `is_empty()` verifica si la lista de frontera esta vacia.
* `pop()` primero verifica si la lista no esta vacia, en caso de que si, lanza una excepción, si no esta elimina el primer elemento ya que es el de mayor prioridad o mejor dicho de menor costo.
* `top()` primero verifica si la lista no esta vacia, en caso de que si, lanza una excepción, si no este retorna el primer elemento, pero al ser una tupla y solo queremos el nodo, solo devolvemos la tercera posición de la tupla que es el nodo como tal.
* `add(node)` agrega el nodo y se ingresa en forma de tupla extrayendo unicamente su atributo de costo y asignandole su contador (indice). Una vez agregado se suma 1 el contador para un posible siguiente elemento.


---




### 2.1.2 Modelen la estructura de datos para un problema de búsqueda:

In [32]:
class Problem:
    def __init__(self, initial_state, goals, graph, heuristics):
        self.initial_state = initial_state
        self.goals = goals
        self.graph = graph
        self.heuristics = heuristics

    def is_goal(self, state):
        return state in self.goals

    def result_actions(self, state):
        actions = self.graph[state]

        return set(actions.keys())

    def action_cost(self, state, action, state_1):
        actions = self.graph[state]
        return actions[state_1]

    def heuristic(self, state):
        return self.heuristics[state]

Para la clase `Problem` creamos la clase para modelar la estructura de datos, en este caso decidimos usar un grafo, esto ya que es la estructura mas comun para este tipo de problemas, asi mismo, para el caso de usar un arbol no binario, ya tenemos definido el atributo padre, para este caso de un grafo el padre seria el nodo anterior.

En esta clase modelamos los atributos necesarios para resolver el problema: el estado 
inicial `initial`, la o las metas `goals`, la estructura de datos que representa el mundo 
(en este caso un grafo) `graph`, y una tabla de heurísticas precalculadas `heuristics`.

Los métodos creados son:

* **is_goal**: indica si un estado dado es una meta (está en `goals`).
* **result_actions**: devuelve el conjunto de estados a los que se puede llegar en un paso desde el estado dado (los vecinos en el grafo).
* **action_cost**: devuelve el costo (float) de moverse de un estado a otro.
* **heuristic**: devuelve el valor estimado (float) de qué tan cerca está un estado de la meta, según la tabla precalculada.

---

## 2.2. B. IMPLEMENTAR

### 2.2.1.Implemente el algoritmo de búsqueda del mejor primero:

In [37]:
# ============================================================
# 1. Represente el nodo de fallo del espacio de búsqueda
# ============================================================
    
FAIL = Node(None)

# ============================================================
# 2. Implemente la función para expandir nodos fronterizos
# ============================================================
def expand(problem, node):
    state = node.state
    neighbors = problem.result_actions(state)
    neighbors = list(neighbors)
    expansion = []
    for neighbor in neighbors:
        cost = problem.action_cost(state, neighbor, neighbor) + node.cost
        node_1 = Node(neighbor, node, neighbor, cost)
        expansion.append(node_1)

    return expansion

    
# ============================================================
# 3. Implemente el algoritmo de búsqueda del mejor primero (Best-First Search)
# ============================================================

def best_first_search(problem, evaluation_function):
    root_node = Node(problem.initial_state, None, None, 0)
    frontier = Frontier(evaluation_function)
    frontier.add(root_node)

    while not frontier.is_empty():
        current_node = frontier.pop()

        if problem.is_goal(current_node.state):
            return current_node

        for neighbor in expand(problem, current_node):
            frontier.add(neighbor)
    return FAIL

Para la implementación del nodo de fallo, entre las opciones que nos dio el profesor, se decidio que sea cuando un nodo sea no tenga un estado definido, o mejor dicho que sea como un nodo vacio o de estado `None`. 

para la función de expandir los nodos fronterizos `expand` definimos el estado y los vecinos los cuales guardamos en una lista, aqui creamos otra lista que sera la de expansion, esta guardara los nodos vecinos que tenemos (aqui tambien calculamos el costo de ir a cada vecino usando la funcion de `action_cost`).

para el algoritmo de GBST o greedy Best first search definimos el nodo raiz que es aquel que sera nuestro punto de partida para la busqieda, asi como traer los nodos de frontera, ahora aqui hacemos un ciclo que mientras la frontera no este vacia, vamos eliminando los nodos que vamos recorriendo ya que los estamos explorando, asi mismo, revisamos que ese nodo que estamos explorando sea un estado meta, en caso que no guardamos los vecinos de ese nodo a la frontera. En caso de no encontrar la meta retornamos que no se puede llegar.

---

### 2.2.2. Pruebe la implementación del algoritmo con el siguiente espacio de ejemplo:

![Grafo.png](attachment:Grafo.png)

In [40]:
# ============================================================
# 1. Implemente la función para visualizar la secuencia de acciones de un `Node`
# usando `Node.representation()`
# ============================================================

def path_actions(node):
    nodes = []
    while True:        
    	nodes.append(node)        
    	if node.parent == None:
    		break
    	node = node.parent

    nodes.reverse()

    actions = []
    for node in nodes:
        action = node.state
        actions.append(action)

    return actions

# ============================================================
# 2. Represent the example search problem
# ============================================================


initial = "S"
goals = {"G"}
graph = {
    "S": {"A": 3, "D": 4},
    "A": {"S": 3, "D": 5, "B": 4},
    "D": {"S": 4, "A": 5, "E": 2},
    "B": {"A": 4, "C": 4, "E": 5},
    "C": {"B": 4},
    "E": {"D": 2, "B": 5, "F": 4},
    "F": {"E": 4, "G": 3},
    "G": {"F": 3}
}
heuristics = {"S": 11, "A": 10.4, "D": 8.9, "B": 6.7, "E": 6.9, "C": 4.0, "F": 3.0, "G": 0}

problem = Problem(initial, goals, graph, heuristics)

# ============================================================
# 3. Use la implementación del algoritmo de búsqueda del mejor primero
# aplicándola al problema de búsqueda de ejemplo
# ============================================================


resultado = best_first_search(problem, lambda node: node.cost)

# ============================================================
# 4. Use la función para visualizar la secuencia de acciones
# con el resultado del algoritmo de búsqueda del mejor primero
# aplicado al problema de búsqueda de ejemplo
# ============================================================


solution = path_actions(resultado)
solution_visual = " -> ".join(solution)
print(solution_visual)

S -> D -> E -> F -> G
